# AMPS II · Week 12 — HW11
## The axisymmetric weak form, second-order kinetics, SUPG, the Graetz problem, Navier–Stokes coupling, Taylor–Aris dispersion and an LLM-generated reactor script

**Due before Week 13 · 100 points.** Submit the executed notebook: *Runtime → Restart session and run all*, fill in the AI-usage log at the end.
Every number you report must come from a cell of this notebook. The first cell of Q7 runs an LLM-generated script that is **meant** to be wrong. The whole HW runs in a few minutes.

| Q | Topic | Points |
|---|---|:---:|
| 1 | Theory: the axisymmetric weak form, Danckwerts, the mixing cup, SUPG consistency | 15 |
| 2 | Second-order kinetics: Newton, plug flow and segregated flow | 14 |
| 3 | SUPG: consistency, the cell length and the outlet | 14 |
| 4 | The Graetz problem and axial diffusion | 13 |
| 5 | Navier–Stokes in the tube: verification and one-way coupling | 15 |
| 6 | Radial diffusion in laminar flow: Taylor–Aris dispersion | 14 |
| 7 | Debugging an LLM-generated reactor script | 15 |

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

The model module from the lab (unchanged). Run the cell once.

In [ ]:
%%writefile /tmp/fnx/reactor.py
"""reactor.py — a tubular reactor, axisymmetric (z, r), DOLFINx 0.11 (AMPS II, Week 12).

Coordinates x[0] = z (axial, 0..L), x[1] = r (radial, 0..1, axis at r = 0); lengths in units of the tube radius R,
velocities in units of the mean velocity U. Every integral carries the weight r (axisymmetry; the 2*pi cancels).
    species:   u . grad c = (1/Pe) lap_axi c - Da c^n          Pe = U R / D, Da = k R c0^(n-1) / U
    flow:      (u . grad) u = -grad p + (1/Re) (lap_axi u - u_r/r^2 e_r),  div_axi u = 0,   Re = U R / nu
Inlet z = 0 (Danckwerts or Dirichlet), outlet z = L (no diffusive flux / do-nothing), wall r = 1 (inert or c = 0), axis r = 0."""
import gc
import numpy as np
import ufl, basix.ufl
from mpi4py import MPI
from dolfinx import fem, mesh
from dolfinx.fem.petsc import NonlinearProblem

comm = MPI.COMM_WORLD
gc.disable()                                   # PETSc destructors are collective: collect at the same point on all ranks
def collect():
    gc.collect()
def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)
LU = {"ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"}
NEWTON = dict(LU, snes_type="newtonls", snes_linesearch_type="none", snes_rtol=1e-10, snes_atol=1e-11, snes_stol=0.0, snes_max_it=30)

def tube_mesh(L=10.0, nz=100, nr=10, grade=0.0):
    """[0, L] x [0, 1] in (z, r), quadrilaterals; grade > 0 clusters cells towards the wall r = 1 (r -> 1 - (1 - s)^(1 + grade))"""
    msh = mesh.create_rectangle(comm, [[0, 0], [L, 1]], [nz, nr], mesh.CellType.quadrilateral)
    if grade > 0:
        x = msh.geometry.x; x[:, 1] = 1 - (1 - x[:, 1]) ** (1 + grade)
    return msh

def _L(msh):
    return comm.allreduce(msh.geometry.x[:, 0].max(), op=MPI.MAX)

def velocity(msh, kind="poiseuille", deg=2):
    """prescribed velocity (u_z, u_r) with mean 1: "plug" (u_z = 1) or "poiseuille" (u_z = 2 (1 - r^2)); a P2 vector Function"""
    V = fem.functionspace(msh, ("Lagrange", deg, (2,))); u = fem.Function(V)
    if kind == "plug":
        u.interpolate(lambda x: np.vstack([np.ones(x.shape[1]), np.zeros(x.shape[1])]))
    else:
        u.interpolate(lambda x: np.vstack([2 * (1 - x[1] ** 2), np.zeros(x.shape[1])]))
    return u

class Flow:
    """steady axisymmetric Navier-Stokes in the tube: uniform inlet u = (1, 0), no slip at r = 1, u_r = 0 on the axis, do-nothing outlet"""
    count = 0
    def __init__(self, msh, Re=10.0, inlet="uniform"):
        collect(); Flow.count += 1
        self.msh = msh; c = msh.basix_cell(); self.L = _L(msh)
        W = self.W = fem.functionspace(msh, basix.ufl.mixed_element([basix.ufl.element("Lagrange", c, 2, shape=(2,)), basix.ufl.element("Lagrange", c, 1)]))
        self.w = fem.Function(W); u, p = ufl.split(self.w); v, q = ufl.TestFunctions(W)
        self.Re = fem.Constant(msh, float(Re)); r = ufl.SpatialCoordinate(msh)[1]
        div = lambda a: a[0].dx(0) + a[1].dx(1) + a[1] / r
        F = (ufl.inner(ufl.grad(u) * u, v) + 1 / self.Re * (ufl.inner(ufl.grad(u), ufl.grad(v)) + u[1] * v[1] / r ** 2)
             - p * div(v) - q * div(u)) * r * ufl.dx
        V, _ = W.sub(0).collapse(); Vr, _ = W.sub(0).sub(1).collapse()
        fdim = 1
        inl = mesh.locate_entities_boundary(msh, fdim, lambda x: np.isclose(x[0], 0))
        wall = mesh.locate_entities_boundary(msh, fdim, lambda x: np.isclose(x[1], 1))
        axis = mesh.locate_entities_boundary(msh, fdim, lambda x: np.isclose(x[1], 0))
        uin = fem.Function(V)
        if inlet == "uniform":
            uin.interpolate(lambda x: np.vstack([np.ones(x.shape[1]), np.zeros(x.shape[1])]))
        else:
            uin.interpolate(lambda x: np.vstack([2 * (1 - x[1] ** 2), np.zeros(x.shape[1])]))
        self.bcs = [fem.dirichletbc(fem.Function(V), fem.locate_dofs_topological((W.sub(0), V), fdim, wall), W.sub(0)),
                    fem.dirichletbc(uin, fem.locate_dofs_topological((W.sub(0), V), fdim, inl), W.sub(0)),
                    fem.dirichletbc(fem.Function(Vr), fem.locate_dofs_topological((W.sub(0).sub(1), Vr), fdim, axis), W.sub(0).sub(1))]
        self.pb = NonlinearProblem(F, self.w, bcs=self.bcs, petsc_options_prefix=f"fl{Flow.count}_", petsc_options=NEWTON)

    def solve(self, Re=None):
        if Re is not None:
            self.Re.value = Re
        self.pb.solve(); s = self.pb.solver
        return int(s.getConvergedReason()), int(s.getIterationNumber())

    def u(self):
        return self.w.sub(0).collapse()

    def p(self):
        return self.w.sub(1).collapse()

class Species:
    """u . grad c = (1/Pe) lap_axi c - Da c^n, axisymmetric; inlet "danckwerts" or "dirichlet" (c = 1), wall "inert" or "reactive" (c = 0)"""
    count = 0
    def __init__(self, msh, u, Pe=100.0, Da=1.0, order=1, inlet="danckwerts", wall="inert", supg=True, supg_reaction=True, hkind="streamline", deg=2):
        collect(); Species.count += 1
        self.msh = msh; self.u = u; self.L = _L(msh); self.order = order
        S = self.S = fem.functionspace(msh, ("Lagrange", deg)); self.c = fem.Function(S); w = ufl.TestFunction(S)
        self.Pe, self.Da = fem.Constant(msh, float(Pe)), fem.Constant(msh, float(Da))
        X = ufl.SpatialCoordinate(msh); r = X[1]; c = self.c; n = ufl.FacetNormal(msh); self.r = r
        fdim = 1
        inl = mesh.locate_entities_boundary(msh, fdim, lambda x: np.isclose(x[0], 0))
        out = mesh.locate_entities_boundary(msh, fdim, lambda x: np.isclose(x[0], self.L))
        wl = mesh.locate_entities_boundary(msh, fdim, lambda x: np.isclose(x[1], 1))
        tags = np.concatenate([inl, out, wl]); vals = np.r_[np.full(len(inl), 1), np.full(len(out), 2), np.full(len(wl), 3)].astype(np.int32)
        o = np.argsort(tags); tags, vals = tags[o], vals[o]
        self.ds = ufl.Measure("ds", domain=msh, subdomain_data=mesh.meshtags(msh, fdim, tags, vals))
        rate = c if order == 1 else c * c if order == 2 else c ** order
        F = (ufl.dot(u, ufl.grad(c)) * w + 1 / self.Pe * ufl.dot(ufl.grad(c), ufl.grad(w)) + self.Da * rate * w) * r * ufl.dx
        self.bcs = []
        if inlet == "danckwerts":                                                     # u c - (1/Pe) dc/dz = u * 1 at z = 0
            F += u[0] * (c - 1.0) * w * r * self.ds(1)
        else:
            one = fem.Function(S); one.x.array[:] = 1.0
            self.bcs.append(fem.dirichletbc(one, fem.locate_dofs_topological(S, fdim, inl)))
        if wall == "reactive":
            self.bcs.append(fem.dirichletbc(fem.Function(S), fem.locate_dofs_topological(S, fdim, wl)))
        if supg:
            un = ufl.sqrt(ufl.dot(u, u) + 1e-30)
            if hkind == "streamline":                                                   # cell length along u: |u| / |J^-1 u|
                Ku = ufl.JacobianInverse(msh) * u; h = un / ufl.sqrt(ufl.dot(Ku, Ku) + 1e-30)
            else:                                                                       # the cell's diameter (diagonal)
                h = ufl.CellDiameter(msh)
            Peh = un * h * self.Pe / (2 * deg)
            tau = h / (2 * deg * un) * (1 / ufl.tanh(Peh) - 1 / Peh)                     # SUPG parameter (cell Peclet with h/deg)
            lap = ufl.div(ufl.grad(c)) + c.dx(1) / r                                    # axisymmetric Laplacian
            res = ufl.dot(u, ufl.grad(c)) - 1 / self.Pe * lap + (self.Da * rate if supg_reaction else 0)
            F += tau * res * ufl.dot(u, ufl.grad(w)) * r * ufl.dx
        self.F = F
        self.pb = NonlinearProblem(F, c, bcs=self.bcs, petsc_options_prefix=f"sp{Species.count}_", petsc_options=NEWTON)
        self.c.x.array[:] = 1.0 if wall == "inert" else 0.5

    def solve(self, Pe=None, Da=None):
        if Pe is not None:
            self.Pe.value = Pe
        if Da is not None:
            self.Da.value = Da
        self.pb.solve(); s = self.pb.solver
        return int(s.getConvergedReason()), int(s.getIterationNumber())

    # ── quantities ─────────────────────────────────────────────────
    def _int(self, e, m):
        return comm.allreduce(fem.assemble_scalar(fem.form(e * m)), op=MPI.SUM)

    def mixing_cup(self):
        """outlet concentration weighted by the flow: int u_z c r dr / int u_z r dr at z = L"""
        return self._int(self.u[0] * self.c * self.r, self.ds(2)) / self._int(self.u[0] * self.r, self.ds(2))

    def area_mean(self):
        """outlet concentration averaged over the cross-section: int c r dr / int r dr"""
        return self._int(self.c * self.r, self.ds(2)) / self._int(self.r, self.ds(2))

    def profile(self, z, n=201):
        """r and c(z, r) on a radial line (every rank gets all values)"""
        P = np.zeros((n, 3)); P[:, 0] = z; P[:, 1] = np.linspace(0, 1, n)
        return P[:, 1], probe(self.msh, self.c, P)

    def mixing_cup_at(self, zs, n=401):
        """mixing-cup concentration at axial positions zs: int u_z c r dr / int u_z r dr on radial lines (Simpson)"""
        Uz = self.u.sub(0).collapse() if hasattr(self.u, "sub") else None
        out = []
        for z in np.atleast_1d(zs):
            P = np.zeros((n, 3)); P[:, 0] = z; P[:, 1] = np.linspace(0, 1, n)
            cr = probe(self.msh, self.c, P); ur = probe(self.msh, Uz, P); r = P[:, 1]
            wts = np.ones(n); wts[1:-1:2] = 4; wts[2:-1:2] = 2
            out.append(np.sum(wts * ur * cr * r) / np.sum(wts * ur * r))
        return np.array(out)

    def sherwood(self, zs, dz=0.05):
        """local Sherwood number (diameter-based) from the decay of the mixing-cup concentration, reactive wall:
        Sh = -Pe d ln(c_mix)/dz (axial diffusion neglected in the balance)"""
        zs = np.atleast_1d(zs); a = self.mixing_cup_at(zs - dz); b = self.mixing_cup_at(zs + dz)
        return -float(self.Pe.value) * (np.log(b) - np.log(a)) / (2 * dz)

def probe(msh, f, P):
    """values of a scalar Function at points P (n x 3) — every rank gets all values"""
    from dolfinx import geometry
    nloc = msh.topology.index_map(2).size_local
    cand = geometry.compute_colliding_cells(msh, geometry.compute_collisions_points(geometry.bb_tree(msh, 2), P), P)
    out, cnt = np.zeros(len(P)), np.zeros(len(P)); idx, cells = [], []
    for j in range(len(P)):
        own = [c for c in cand.links(j) if c < nloc]
        if own:
            idx.append(j); cells.append(own[0])
    if idx:
        val = np.asarray(f.eval(P[idx], np.array(cells, dtype=np.int32))).reshape(len(idx), -1)
        out[idx] = val[:, 0]; cnt[idx] = 1
    return comm.allreduce(out, op=MPI.SUM) / np.maximum(comm.allreduce(cnt, op=MPI.SUM), 1)

# exact references ─────────────────────────────────────────────────────────────
def danckwerts(PeL, DaL):
    """plug flow with axial dispersion, 1st order, Danckwerts inlet: outlet c (Pe_L = U L / D, Da_L = k L / U)"""
    a = np.sqrt(1 + 4 * DaL / PeL)
    return 4 * a * np.exp(PeL / 2 * (1 - a)) / ((1 + a) ** 2 - (1 - a) ** 2 * np.exp(-a * PeL))

def dirichlet_dispersion(PeL, DaL):
    """plug flow with axial dispersion, 1st order, c = 1 at the inlet, dc/dz = 0 at the outlet: outlet c"""
    m1 = PeL / 2 * (1 + np.sqrt(1 + 4 * DaL / PeL)); m2 = PeL / 2 * (1 - np.sqrt(1 + 4 * DaL / PeL))
    # c = A e^{m1 s} + B e^{m2 s}, s in [0, 1]: A + B = 1, A m1 e^{m1} + B m2 e^{m2} = 0
    # => A e^{m1} = -B (m2 / m1) e^{m2}: c(1) = B e^{m2} (1 - m2 / m1). (Do NOT form A = 1 - B: for large Pe_L, B = 1 to round-off,
    #    A becomes 0 and the term A e^{m1} = O(e^{m2} Da_L / Pe_L) is lost — 1 % at Pe_L = 200.)
    B = 1 / (1 - m2 / m1 * np.exp(m2 - m1))
    return B * np.exp(m2) * (1 - m2 / m1)

def taylor_aris(Pe, L, Da):
    """laminar flow with radial diffusion, long tube: plug flow with the Taylor-Aris dispersion D_eff = D (1 + Pe^2 / 48),
    Pe = U R / D; returns the outlet c of the Danckwerts dispersion model"""
    Pe_eff = Pe / (1 + Pe ** 2 / 48)
    return danckwerts(Pe_eff * L, Da * L)

def segregated(DaL, order=1, n=200001):
    """laminar (Poiseuille) flow without diffusion: every streamline a plug-flow reactor with residence time L / u(r); mixing cup"""
    r = np.linspace(0, 1, n)[:-1] + 0.5 / n; u = 2 * (1 - r ** 2); t = DaL / u
    c = np.exp(-t) if order == 1 else 1 / (1 + t)
    return np.sum(u * c * 2 * r) / n

def graetz(nev=10, N=800):
    """Graetz problem: 2 (1 - r^2) dc/dzeta = (1/r)(r c_r)_r, c(1) = 0, c(r, 0) = 1, zeta = z / Pe (axial diffusion neglected).
    Finite volumes in r -> eigenvalues w_k and mixing-cup weights m_k: c_mix(zeta) = sum_k m_k exp(-w_k zeta); Sh_inf = w_1 = 3.657"""
    h = 1.0 / N; r = (np.arange(N) + 0.5) * h; rp = r + h / 2; rm = r - h / 2
    A = np.zeros((N, N))
    for i in range(N):
        A[i, i] = -(rp[i] + rm[i]) / h ** 2
        if i > 0:
            A[i, i - 1] = rm[i] / h ** 2
        if i < N - 1:
            A[i, i + 1] = rp[i] / h ** 2
    A[N - 1, N - 1] = -(rm[N - 1] / h ** 2 + rp[N - 1] / (h / 2) / h)      # c = 0 at r = 1 (half cell)
    Bm = np.diag(2 * (1 - r ** 2) * r)
    from numpy.linalg import eig
    w, V = eig(np.linalg.solve(Bm, -A))
    o = np.argsort(w.real); w = w.real[o][:nev]; V = V.real[:, o][:, :nev]
    # coefficients for c(r, 0) = 1 (weight 2(1 - r^2) r)
    wt = 2 * (1 - r ** 2) * r * h
    coef = np.array([np.sum(wt * V[:, k]) / np.sum(wt * V[:, k] ** 2) for k in range(nev)])
    # mixing cup c(zeta) = sum coef_k <u phi_k> exp(-w_k zeta) / <u>, <u> = int 2(1-r^2) r dr = 1/2
    mc = np.array([np.sum(wt * V[:, k]) for k in range(nev)]) * coef / 0.5
    return w, mc

## Q1 — Theory (15 pts)

**(a) (5)** Write $\int_\Omega(\mathbf u\cdot\nabla c\,w+D\nabla c\cdot\nabla w+k\,c\,w)\,dV$ for an axisymmetric problem in cylindrical coordinates $(z,r,\theta)$ and reduce it to an integral over the half-plane $(z,r)$. What happens to $dV$, to $\nabla c\cdot\nabla w$
(no swirl), and why is no boundary condition needed on the axis $r=0$?

**(b) (4)** Derive the Danckwerts inlet condition from a flux balance across $z=0$ and show that it enters the weak form as $\int_\text{inlet}u_z(c-c_0)\,w\,r\,dr$ (dimensionless).

**(c) (3)** Show that the mixing-cup concentration at the outlet is what a species balance over the reactor needs: inflow − outflow = consumption, with the outflow $\int u_zc\,r\,dr$ (inert wall, no axial diffusion through the outlet).

**(d) (3)** SUPG adds $\sum_K\int_K\tau\,R(c_h)\,\mathbf u\cdot\nabla w\,r\,dx$. Write $R$ for this problem (axisymmetric Laplacian, $n$-th order reaction) and explain "consistent".

> *your answer*

## Q2 — Second-order kinetics: Newton, plug flow and segregated flow (14 pts)

$r=k\,c^2$, `order=2`; Newton solves the nonlinear problem.

**(a) (7)** Plug flow ($Pe=10^4$, $L=10$, 100 × 2 cells) for $Da_L=1,10,100$ against $1/(1+Da_L)$; report Newton's iterations.

**(b) (7)** Segregated laminar flow ($Pe=10^8$, 100 × 20) for $Da_L=1,10$ against $\int u/(1+Da_L/u)\,2r\,dr$. Compare the laminar penalty (laminar/plug outlet ratio) with first order at the same $Da_L$.

In [ ]:
%%fenicsx -np 2
# TODO: Q2 — second-order kinetics, plug and segregated flow

> *your answer (Q2)*

## Q3 — SUPG: consistency, the cell length and the outlet (14 pts)

Plug flow, $L=4$, 2 cells across the radius (stretched cells for large $n_z$).

**(a) (8)** For $Pe=10^6$, $Da=20$ and $n_z=20,40,80,160$: the maximum error against $e^{-Da z}$ on $r=0.5$ for Galerkin, SUPG, SUPG without the reaction in the residual, and the same with `hkind="diameter"`. Observed orders.

**(b) (6)** For $Pe=10^4$, $Da=2$ ($Da_L=8$): the outlet $c$ of the same four variants at $n_z=20,40,80$ against the dispersion model. Which variant would pass "the outlet agrees with plug flow within 0.1 % of the inlet concentration"? What does that say about such checks?

In [ ]:
%%fenicsx -np 2
# TODO: Q3 — four SUPG variants, profile error and outlet

> *your answer (Q3)*

## Q4 — The Graetz problem and axial diffusion (13 pts)

Reactive wall, Poiseuille flow, $L=20$, graded mesh (200 × 20, `grade=1.0`).

**(a) (7)** For $Pe=10,100,1000$: the outlet mixing cup and the outlet Sherwood number (at $z=19$) against the Graetz series at the same $\zeta$.

**(b) (6)** Explain the trend of the deviation with $Pe$. Which dimensionless group tells you whether the Graetz series (no axial diffusion) is a valid reference?

In [ ]:
%%fenicsx -np 2
# TODO: Q4 — Graetz with axial diffusion, Pe = 10, 100, 1000

> *your answer (Q4)*

## Q5 — Navier–Stokes in the tube: verification and one-way coupling (15 pts)

`Flow(msh, Re)`: uniform inlet $u=(1,0)$, no slip, do-nothing outlet; $L=30$, 150 × 16.

**(a) (7)** For $Re=10$ and 50: the outlet profile against $2(1-r^2)$, $dp/dz$ in the developed region against $-8/Re$, and the length after which the centreline velocity reaches 99 % of 2. Compare with $L_e\approx0.24\,Re\,R$.

**(b) (8)** Use the computed velocity in `Species` ($Pe=100$, $Da=0.05$, inert wall) and compare the outlet mixing cup with Poiseuille and plug flow. Explain the size and the sign of the difference. When would the coupling be two-way?

In [ ]:
%%fenicsx -np 2
# TODO: Q5 — developing flow and one-way coupling

> *your answer (Q5)*

## Q6 — Radial diffusion in laminar flow: Taylor–Aris dispersion (14 pts)

Poiseuille flow, inert wall, $Da_L=1$. Radial diffusion mixes the streamlines; for long tubes the cross-section-averaged concentration obeys a dispersion model with $D_\text{eff}=D(1+Pe^2/48)$ (Taylor–Aris, $Pe=UR/D$).

**(a) (8)** For $(Pe,L)=(2,100),(5,100),(20,100),(20,400),(100,400)$ (2 cells per unit length, 12 across): FE outlet against Taylor–Aris + Danckwerts, segregated flow and plug flow.

**(b) (6)** When is Taylor–Aris a valid reference (compare $L/R$ with $Pe$)? Why does more diffusion (smaller $Pe$) bring the laminar reactor **closer** to plug flow?

In [ ]:
%%fenicsx -np 2
# TODO: Q6 — Taylor-Aris against FE

> *your answer (Q6)*

## Q7 — Debugging an LLM-generated reactor script (15 pts)

An LLM was asked for "a DOLFINx 0.11 script for a laminar-flow tubular reactor (axisymmetric, radius $R=1$, $L=100R$, $Pe=UR/D=5$, first order with $Da=kR/U=0.01$) that computes the conversion and verifies it". It returned the script below.

**(a) (3)** Run it as received (`-np 2`). What does it claim?

**(b) (7)** Find the defects. For each: what it does to the result, whether the script's own check can see it, and a test that exposes it.

**(c) (5)** Correct it (the module may be used) and verify it against a reference with a tolerance that matches the discretisation error.

In [ ]:
%%fenicsx -np 2
# LLM-generated (as received): laminar-flow tubular reactor, first-order reaction, axisymmetric (z, r)
import numpy as np, ufl
from mpi4py import MPI
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem
comm = MPI.COMM_WORLD
L, Pe, Da = 100.0, 5.0, 0.01                  # tube length / R, Pe = U R / D, Da = k R / U  (Da L = 1)
msh = mesh.create_rectangle(comm, [[0, 0], [L, 1]], [200, 12], mesh.CellType.quadrilateral)
V = fem.functionspace(msh, ("Lagrange", 2))
c, w = ufl.TrialFunction(V), ufl.TestFunction(V)
z, r = ufl.SpatialCoordinate(msh)
u = ufl.as_vector([2 * (1 - r ** 2), 0])     # Poiseuille profile, mean velocity 1
a = (ufl.dot(u, ufl.grad(c)) * w + 1 / Pe * ufl.dot(ufl.grad(c), ufl.grad(w)) + Da * c * w) * ufl.dx
Lf = fem.Constant(msh, 0.0) * w * ufl.dx
inlet = fem.locate_dofs_geometrical(V, lambda x: np.isclose(x[0], 0.0))
one = fem.Function(V); one.x.array[:] = 1.0
c_h = LinearProblem(a, Lf, bcs=[fem.dirichletbc(one, inlet)], petsc_options_prefix="lfr_",
                    petsc_options={"ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"}).solve()
outlet = mesh.locate_entities_boundary(msh, 1, lambda x: np.isclose(x[0], L))
ds = ufl.Measure("ds", domain=msh, subdomain_data=mesh.meshtags(msh, 1, np.sort(outlet), np.ones(len(outlet), dtype=np.int32)))
c_out = comm.allreduce(fem.assemble_scalar(fem.form(c_h * ds(1))), op=MPI.SUM)        # mean outlet concentration (outlet length 1)
X = 1 - c_out
X_pfr = 1 - np.exp(-Da * L)                                                           # plug flow
rr = (np.arange(100000) + 0.5) / 100000; uu = 2 * (1 - rr ** 2)
X_lfr = 1 - np.sum(uu * np.exp(-Da * L / uu) * 2 * rr) / 100000                      # laminar flow, no diffusion (segregated)
ok = X_lfr - 0.05 <= X <= X_pfr + 0.05                                                # allow for discretisation error
print(f"conversion X = {X:.4f};  bounds: segregated laminar {X_lfr:.4f} <= X <= plug flow {X_pfr:.4f}  ->  {'verified' if ok else 'CHECK'}")

In [ ]:
%%fenicsx -np 2
# TODO: Q7(c) — the corrected script and its verification

> *your list of defects and tests (Q7b), and your verification (Q7c)*

## 🤖 AI-usage log

*Required in every assignment and project. Using an LLM is encouraged and never penalized — an empty or vague log is.*

| # | Tool / model | What I asked (key prompt, shortened) | What it got wrong or missed | How I detected it | How I fixed it |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |

**Code provenance** — which cells/functions were mostly generated, which did you write or substantially change?

> …

**Verification** — how do you know the final result is correct (MMS rates, benchmark value, conservation check, parallel = serial)?

> …

**Reflection (2–3 sentences)** — what would you do differently when prompting next time?

> …